# Módulo 01 · Aula 05 — Arquivos, Erros e Debug

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Até aqui os dados estavam escritos no código. Chega. A partir de agora eles vêm de arquivos — e arquivos falham: não existem, estão corrompidos, têm encoding errado, têm uma linha com "n/d" no meio dos números.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | Exceções e tratamento | Seu programa não pode morrer por uma linha ruim |
| 2 | `try/except/else/finally` | Anatomia completa |
| 3 | `raise` e exceções próprias | Falhar com mensagem útil |
| 4 | `pathlib` | Caminhos que funcionam em Windows e Linux |
| 5 | Arquivos de texto e `with` | Abrir e fechar corretamente |
| 6 | CSV | O formato do dia a dia |
| 7 | JSON | O formato das APIs |
| 8 | Debug no VS Code | Parar de depurar com `print` |

## 1. Erros: sintaxe vs exceção

- **Erro de sintaxe** — o código nem chega a rodar. Python não entendeu o que você escreveu.
- **Exceção** — o código roda e falha **durante** a execução.

Quando uma exceção não é tratada, Python imprime o *traceback* e encerra o programa.

### Lendo um traceback

Leia **de baixo para cima**:

1. A **última linha** diz o tipo do erro e a mensagem. É a informação mais importante.
2. Acima dela, a pilha de chamadas — a linha **mais embaixo** é onde estourou; as de cima, quem chamou.

In [ ]:
# Provoque o erro e leia o traceback com atenção
def dividir(a, b):
    return a / b


def calcular_ticket_medio(faturamento, pedidos):
    return dividir(faturamento, pedidos)


try:
    calcular_ticket_medio(10000, 0)
except ZeroDivisionError:
    import traceback
    traceback.print_exc()

### Exceções que você vai encontrar

| Exceção | Quando acontece |
|---------|-----------------|
| `ValueError` | Tipo certo, valor inválido: `int("abc")` |
| `TypeError` | Tipo errado: `"3" + 5` |
| `KeyError` | Chave inexistente em `dict` |
| `IndexError` | Índice fora do intervalo em lista |
| `FileNotFoundError` | Arquivo não existe |
| `PermissionError` | Sem permissão de acesso |
| `ZeroDivisionError` | Divisão por zero |
| `AttributeError` | Método/atributo não existe (`None.strip()`) |
| `ImportError` / `ModuleNotFoundError` | Import falhou |
| `UnicodeDecodeError` | Encoding errado ao ler arquivo |

Todas herdam de `Exception`, que herda de `BaseException`.

In [ ]:
casos = [
    ("int('abc')",          lambda: int("abc")),
    ("'3' + 5",             lambda: "3" + 5),
    ("{'a':1}['b']",        lambda: {"a": 1}["b"]),
    ("[1,2,3][10]",         lambda: [1, 2, 3][10]),
    ("1/0",                 lambda: 1 / 0),
    ("None.strip()",        lambda: None.strip()),
]

for descricao, funcao in casos:
    try:
        funcao()
    except Exception as erro:
        print(f"{descricao:<18} -> {type(erro).__name__}: {erro}")

## 2. `try` / `except` / `else` / `finally`

```python
try:
    # código que PODE falhar
except TipoDoErro as e:
    # o que fazer se falhar
else:
    # roda SÓ se NÃO houve exceção
finally:
    # roda SEMPRE (com ou sem erro) — limpeza
```

**Regras de bom uso:**

1. Capture o **tipo específico**, não `Exception` genérico.
2. **Nunca** use `except: pass` — engolir erro silenciosamente é como desligar o alarme de incêndio.
3. Mantenha o bloco `try` **curto** — só a linha que pode falhar.
4. Se não sabe o que fazer com o erro, deixe-o subir.

In [ ]:
def converter_para_float(texto, padrao=0.0):
    """Converte texto para float, devolvendo um padrão em caso de falha."""
    try:
        return float(texto)
    except (ValueError, TypeError):
        return padrao


entradas = ["120.50", "89,90", "", None, "n/d", "45", "1.5e3"]

for e in entradas:
    print(f"{repr(e):<10} -> {converter_para_float(e)}")

In [ ]:
# Anatomia completa
def processar(valor):
    print(f"\n--- processando {repr(valor)} ---")
    try:
        numero = float(valor)
        resultado = 100 / numero
    except ValueError as e:
        print(f"  except  : valor não numérico ({e})")
        return None
    except ZeroDivisionError:
        print("  except  : divisão por zero")
        return None
    else:
        print(f"  else    : sucesso, resultado = {resultado:.2f}")
        return resultado
    finally:
        print("  finally : sempre executa (fechar conexões, liberar recursos)")


processar("4")
processar("abc")
processar("0")

In [ ]:
# ANTIPADRÕES — não faça isso
dados = ["10", "abc", "30"]

# ❌ Engolir tudo em silêncio: você nunca saberá o que deu errado
total = 0
for d in dados:
    try:
        total += int(d)
    except:
        pass
print("Silencioso (você não sabe que perdeu dados):", total)

# ✅ Capturar o específico e REGISTRAR
total = 0
descartados = []
for d in dados:
    try:
        total += int(d)
    except ValueError:
        descartados.append(d)

print("Correto:", total, "| descartados:", descartados)

## 3. `raise` — levantando exceções

Levante uma exceção quando a função recebe algo que **não sabe processar**. Falhar cedo, com mensagem clara, é melhor que retornar um valor errado silenciosamente.

Para regras de negócio, crie **exceções próprias**: elas comunicam intenção e permitem tratamento específico lá em cima.

In [ ]:
def calcular_desconto(valor, percentual):
    """Valida as entradas antes de calcular."""
    if not isinstance(valor, (int, float)):
        raise TypeError(f"valor deve ser numérico, recebido {type(valor).__name__}")
    if valor < 0:
        raise ValueError(f"valor não pode ser negativo: {valor}")
    if not 0 <= percentual <= 1:
        raise ValueError(f"percentual deve estar entre 0 e 1, recebido {percentual}")
    return round(valor * percentual, 2)


print(calcular_desconto(1000, 0.15))

for args in [(1000, 1.5), (-50, 0.1), ("mil", 0.1)]:
    try:
        calcular_desconto(*args)
    except (ValueError, TypeError) as e:
        print(f"{str(args):<16} -> {type(e).__name__}: {e}")

In [ ]:
# Exceções próprias — hierarquia de domínio
class AtlasError(Exception):
    """Exceção base de todo o sistema Atlas."""


class PedidoInvalidoError(AtlasError):
    """O pedido não passou na validação de negócio."""


class EstoqueInsuficienteError(AtlasError):
    """Não há estoque para atender o pedido."""

    def __init__(self, produto, solicitado, disponivel):
        self.produto = produto
        self.solicitado = solicitado
        self.disponivel = disponivel
        super().__init__(
            f"{produto}: solicitado {solicitado}, disponível {disponivel}"
        )


ESTOQUE = {"Notebook": 3, "Mouse": 150, "Teclado": 0}


def reservar(produto, quantidade):
    if quantidade <= 0:
        raise PedidoInvalidoError(f"quantidade inválida: {quantidade}")
    disponivel = ESTOQUE.get(produto)
    if disponivel is None:
        raise PedidoInvalidoError(f"produto inexistente: {produto}")
    if quantidade > disponivel:
        raise EstoqueInsuficienteError(produto, quantidade, disponivel)
    return f"✅ Reservado: {quantidade}x {produto}"


for produto, qtd in [("Notebook", 2), ("Notebook", 10), ("Teclado", 1), ("Drone", 1), ("Mouse", 0)]:
    try:
        print(reservar(produto, qtd))
    except EstoqueInsuficienteError as e:
        print(f"⚠️  Estoque: {e} (faltam {e.solicitado - e.disponivel})")
    except AtlasError as e:
        print(f"❌ {type(e).__name__}: {e}")

> 💡 **Por que uma exceção base (`AtlasError`)?** Porque lá no topo do sistema você pode escrever `except AtlasError:` e capturar **qualquer** erro de negócio de uma vez, sem capturar bugs de programação (que devem estourar e aparecer).

## 4. `pathlib` — caminhos que não quebram

O jeito antigo era concatenar strings: `"dados" + "/" + "vendas.csv"`. Isso quebra no Windows (`\` vs `/`).

O jeito moderno é `pathlib.Path`, que usa o operador `/` e resolve o separador correto para cada sistema operacional.

In [ ]:
from pathlib import Path

base = Path("dados")
arquivo = base / "vendas" / "2026-08.csv"

print("Caminho completo :", arquivo)
print("Nome             :", arquivo.name)
print("Só o nome        :", arquivo.stem)
print("Extensão         :", arquivo.suffix)
print("Pasta pai        :", arquivo.parent)
print("Absoluto         :", arquivo.resolve())
print()
print("Existe?          :", arquivo.exists())
print("Pasta atual      :", Path.cwd())

In [ ]:
# Criando estrutura de pastas
base = Path("dados_aula")
base.mkdir(parents=True, exist_ok=True)      # não reclama se já existir
(base / "brutos").mkdir(exist_ok=True)
(base / "processados").mkdir(exist_ok=True)

print("Criado:", base.resolve())
print("Conteúdo:", [p.name for p in base.iterdir()])

## 5. Arquivos de texto e o `with`

```python
with open(caminho, modo, encoding="utf-8") as f:
    ...
```

O `with` é um **context manager**: ele garante que o arquivo seja fechado, mesmo se ocorrer uma exceção no meio. **Sempre use `with`.**

### Modos de abertura

| Modo | Significado |
|------|-------------|
| `"r"` | Leitura (padrão). Erro se não existir |
| `"w"` | Escrita. **Apaga o conteúdo** se existir |
| `"a"` | Append — acrescenta ao fim |
| `"x"` | Criação exclusiva. Erro se já existir |
| `"b"` | Sufixo binário (`"rb"`, `"wb"`) |

⚠️ **Sempre passe `encoding="utf-8"`.** O padrão varia por sistema (no Windows costuma ser `cp1252`), e é a causa nº 1 de acentos virarem `Ã§Ã£o`.

In [ ]:
from pathlib import Path

base = Path("dados_aula")
log = base / "relatorio.txt"

# ESCRITA — 'w' sobrescreve
with open(log, "w", encoding="utf-8") as f:
    f.write("RELATÓRIO AURORA COMÉRCIO\n")
    f.write("Período: Agosto/2026\n")
    f.write("-" * 30 + "\n")
    f.writelines([f"Linha {i}\n" for i in range(1, 4)])

# APPEND — 'a' acrescenta
with open(log, "a", encoding="utf-8") as f:
    f.write("Gerado automaticamente pelo Atlas.\n")

print(log.read_text(encoding="utf-8"))

In [ ]:
# LEITURA — três formas
caminho = Path("dados_aula/relatorio.txt")

# 1) tudo de uma vez (cuidado com arquivos grandes)
with open(caminho, encoding="utf-8") as f:
    conteudo = f.read()
print("read()      :", len(conteudo), "caracteres")

# 2) lista de linhas
with open(caminho, encoding="utf-8") as f:
    linhas = f.readlines()
print("readlines() :", len(linhas), "linhas")

# 3) linha a linha — PREFIRA ESTA: usa memória constante
print("\nIterando (a forma correta para arquivos grandes):")
with open(caminho, encoding="utf-8") as f:
    for numero, linha in enumerate(f, start=1):
        print(f"  {numero:>2}: {linha.rstrip()}")

In [ ]:
# Atalhos do pathlib para arquivos pequenos
p = Path("dados_aula/nota.txt")

p.write_text("Aurora Comércio\nCampinas-SP\n", encoding="utf-8")
print(p.read_text(encoding="utf-8"))

print("Tamanho:", p.stat().st_size, "bytes")

## 6. CSV — o formato do dia a dia

CSV é texto puro com valores separados. Parece trivial, mas tem armadilhas: campos com vírgula dentro, aspas, quebras de linha. **Use o módulo `csv`, não `.split(",")`.**

### `reader` vs `DictReader`

| | Devolve | Quando usar |
|---|---------|-------------|
| `csv.reader` | lista por linha | Arquivos sem cabeçalho |
| `csv.DictReader` | dict por linha | ✅ Quase sempre — acesso por nome de coluna |

⚠️ **Tudo que sai do CSV é `str`.** Converta números explicitamente.

⚠️ No Windows, abra com `newline=""` para evitar linhas em branco extras.

In [ ]:
import csv
from pathlib import Path

base = Path("dados_aula")
csv_path = base / "vendas.csv"

linhas = [
    ["id", "data", "cidade", "produto", "quantidade", "preco_unitario", "status"],
    [1001, "2026-08-01", "Campinas",       "Notebook", 2,  2599.90, "pago"],
    [1002, "2026-08-01", "São Paulo",      "Mouse",    10, 89.90,   "pago"],
    [1003, "2026-08-02", "Campinas",       "Teclado",  3,  249.00,  "cancelado"],
    [1004, "2026-08-02", "Sorocaba",       "Monitor",  1,  1199.00, "pago"],
    [1005, "2026-08-03", "São Paulo",      "Notebook", 1,  2599.90, "pago"],
    [1006, "2026-08-03", "Campinas",       "Monitor",  4,  1199.00, "pago"],
    [1007, "2026-08-04", "Ribeirão Preto", "Mouse",    25, 89.90,   "pago"],
    [1008, "2026-08-04", "São Paulo",      "Teclado",  6,  249.00,  "pendente"],
    [1009, "2026-08-05", "Campinas",       "Notebook", 1,  2599.90, "pago"],
    [1010, "2026-08-05", "Sorocaba",       "Mouse",    8,  89.90,   "pago"],
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(linhas)

print(csv_path.read_text(encoding="utf-8"))

In [ ]:
# LEITURA com DictReader
with open(csv_path, newline="", encoding="utf-8") as f:
    leitor = csv.DictReader(f)
    registros = list(leitor)

print("Colunas:", list(registros[0].keys()))
print("Registros:", len(registros))
print("\nPrimeiro registro:", registros[0])
print("\n⚠️ Note o tipo:", type(registros[0]["quantidade"]).__name__, "-> tudo é string!")

In [ ]:
# Conversão de tipos + tratamento de linhas ruins
def carregar_vendas(caminho):
    """Lê o CSV convertendo tipos. Devolve (registros_ok, erros)."""
    registros, erros = [], []
    with open(caminho, newline="", encoding="utf-8") as f:
        for numero, linha in enumerate(csv.DictReader(f), start=2):   # 2 = 1ª linha de dados
            try:
                registros.append({
                    "id": int(linha["id"]),
                    "data": linha["data"],
                    "cidade": linha["cidade"].strip().title(),
                    "produto": linha["produto"].strip(),
                    "quantidade": int(linha["quantidade"]),
                    "preco_unitario": float(linha["preco_unitario"]),
                    "status": linha["status"].strip().lower(),
                })
            except (ValueError, KeyError) as e:
                erros.append({"linha": numero, "erro": f"{type(e).__name__}: {e}", "dado": linha})
    return registros, erros


vendas, erros = carregar_vendas(csv_path)
print(f"Carregados: {len(vendas)} | Erros: {len(erros)}")
print(vendas[0])

In [ ]:
# Testando com um CSV SUJO — o caso real
sujo = base / "vendas_sujo.csv"
sujo.write_text(
    "id,data,cidade,produto,quantidade,preco_unitario,status\n"
    "2001,2026-08-01,Campinas,Notebook,2,2599.90,pago\n"
    "2002,2026-08-01,São Paulo,Mouse,dez,89.90,pago\n"          # quantidade inválida
    "2003,2026-08-02,Sorocaba,Teclado,3,,pago\n"                 # preço vazio
    "2004,2026-08-02,  campinas  ,Monitor,1,1199.00,PAGO\n"      # sujeira de texto
    "2005,2026-08-03,Santos,Webcam,abc,n/d,pago\n",              # tudo errado
    encoding="utf-8",
)

registros, erros = carregar_vendas(sujo)

print(f"✅ Válidos: {len(registros)}")
for r in registros:
    print("   ", r["id"], r["cidade"], r["produto"])

print(f"\n❌ Rejeitados: {len(erros)}")
for e in erros:
    print(f"    linha {e['linha']}: {e['erro']}")

In [ ]:
# ESCRITA com DictWriter — gerando um relatório agregado
from collections import defaultdict

por_cidade = defaultdict(float)
for v in vendas:
    if v["status"] == "pago":
        por_cidade[v["cidade"]] += v["quantidade"] * v["preco_unitario"]

saida = base / "faturamento_por_cidade.csv"
with open(saida, "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=["cidade", "faturamento"])
    escritor.writeheader()
    for cidade, total in sorted(por_cidade.items(), key=lambda kv: -kv[1]):
        escritor.writerow({"cidade": cidade, "faturamento": round(total, 2)})

print(saida.read_text(encoding="utf-8"))

> 💡 **Delimitadores e o Excel brasileiro.** O Excel em PT-BR usa `;` como separador (porque a vírgula é o decimal). Passe `delimiter=";"` para `reader`/`writer` quando for esse o caso. Em bases europeias você também verá `encoding="latin-1"`.

## 7. JSON — o formato das APIs

JSON é o formato universal de troca de dados. É o que sua API FastAPI vai devolver no Módulo 06.

| Função | O que faz | Mnemônico |
|--------|-----------|-----------|
| `json.dump(obj, f)` | Python → arquivo | **dump** para arquivo |
| `json.dumps(obj)` | Python → string | **s** de *string* |
| `json.load(f)` | arquivo → Python | **load** de arquivo |
| `json.loads(s)` | string → Python | **s** de *string* |

### Mapeamento de tipos

| Python | JSON |
|--------|------|
| `dict` | object |
| `list`, `tuple` | array |
| `str` | string |
| `int`, `float` | number |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

⚠️ JSON **não** tem tipo data. Datas viram string (padrão ISO: `"2026-08-12"`). Chaves de objeto são **sempre** string.

In [ ]:
import json

relatorio = {
    "empresa": "Aurora Comércio",
    "periodo": "2026-08",
    "gerado_em": "2026-08-12T09:30:00",
    "totais": {
        "faturamento": round(sum(por_cidade.values()), 2),
        "pedidos": len([v for v in vendas if v["status"] == "pago"]),
    },
    "por_cidade": [
        {"cidade": c, "faturamento": round(v, 2)}
        for c, v in sorted(por_cidade.items(), key=lambda kv: -kv[1])
    ],
    "observacoes": None,
}

# Para string (útil para inspecionar)
texto = json.dumps(relatorio, ensure_ascii=False, indent=2)
print(texto)

In [ ]:
# Gravando e lendo de volta
json_path = base / "relatorio.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

with open(json_path, encoding="utf-8") as f:
    recuperado = json.load(f)

print("Empresa:", recuperado["empresa"])
print("Faturamento:", recuperado["totais"]["faturamento"])
print("Melhor praça:", recuperado["por_cidade"][0]["cidade"])
print("\nIguais?", relatorio == recuperado)

> ⚠️ **`ensure_ascii=False` é obrigatório em português.** Sem ele, `"São Paulo"` vira `"São Paulo"` no arquivo — tecnicamente válido, mas ilegível.

In [ ]:
# JSON quebrado -> JSONDecodeError
ruim = '{"cidade": "Campinas", "valor": 100,}'    # vírgula sobrando

try:
    json.loads(ruim)
except json.JSONDecodeError as e:
    print(f"JSONDecodeError: {e.msg}")
    print(f"  linha {e.lineno}, coluna {e.colno}, posição {e.pos}")

In [ ]:
# Objetos que o JSON não conhece -> default=
from datetime import date, datetime
from decimal import Decimal


def serializador(obj):
    """Diz ao json como converter tipos que ele não conhece."""
    if isinstance(obj, (date, datetime)):
        return obj.isoformat()
    if isinstance(obj, Decimal):
        return float(obj)
    raise TypeError(f"Tipo não serializável: {type(obj).__name__}")


dados = {"data": date(2026, 8, 12), "valor": Decimal("1234.56")}

print(json.dumps(dados, default=serializador, ensure_ascii=False, indent=2))

## 8. Depuração no VS Code

Parar de depurar com `print` é um marco na carreira. O depurador mostra **todas** as variáveis a cada passo, sem sujar o código.

### Como usar em um notebook

1. Clique na **margem esquerda** de uma linha de código para colocar um **breakpoint** (bolinha vermelha).
2. Clique na setinha ao lado do botão ▶ da célula e escolha **Debug Cell** (ou `Ctrl+Shift+Alt+Enter`).
3. A execução para no breakpoint. Use o painel **Run and Debug** (`Ctrl+Shift+D`).

### Como usar em um arquivo `.py`

1. Abra o arquivo, coloque breakpoints.
2. `F5` → escolha **Python File**.

### Comandos do depurador

| Tecla | Comando | O que faz |
|-------|---------|-----------|
| `F5` | Continue | Vai até o próximo breakpoint |
| `F10` | Step Over | Executa a linha inteira |
| `F11` | Step Into | **Entra** na função chamada |
| `Shift+F11` | Step Out | Sai da função atual |
| `Ctrl+Shift+F5` | Restart | Reinicia |
| `Shift+F5` | Stop | Encerra |

### Painéis

- **Variables** — todas as variáveis vivas no escopo atual.
- **Watch** — expressões que você quer monitorar (ex.: `len(registros)`, `total > 1000`).
- **Call Stack** — quem chamou quem até chegar aqui.
- **Debug Console** — um REPL **no contexto pausado**: digite qualquer expressão e inspecione.

### Breakpoint condicional

Clique com o **botão direito** na margem → *Add Conditional Breakpoint*. Ex.: `linha["id"] == 2003`. O programa só para quando a condição for verdadeira — essencial para achar o registro problemático num laço de 10.000 iterações.

In [ ]:
# breakpoint() — pausa via código, funciona em qualquer lugar.
# Descomente a linha do breakpoint e rode em "Debug Cell" para experimentar.

def analisar_pedidos(pedidos):
    total = 0.0
    problemas = []

    for p in pedidos:
        # breakpoint()          # <- descomente e rode com Debug Cell
        valor = p["quantidade"] * p["preco_unitario"]
        if valor > 5000:
            problemas.append(p["id"])
        total += valor

    return total, problemas


total, problemas = analisar_pedidos([v for v in vendas if v["status"] == "pago"])
print(f"Total: R$ {total:,.2f}")
print("Pedidos acima de 5k:", problemas)

### As ferramentas de investigação rápida

Nem tudo exige o depurador. Estas três resolvem 80% das dúvidas na hora:

- **`type(x)`** — que tipo é isso?
- **`dir(x)`** — que métodos tem?
- **`repr(x)`** — como isso é *exatamente*? (mostra aspas, `\n`, espaços — `print` esconde)

E, para conferir invariantes durante o desenvolvimento, use `assert`.

In [ ]:
valor = "  120.50\n"

print("print :", valor)
print("repr  :", repr(valor), "   <- agora você VÊ os espaços e o \\n")
print("type  :", type(valor).__name__)
print("métodos úteis:", [m for m in dir(valor) if m.startswith("s")][:8])

In [ ]:
# assert: documenta e verifica pressupostos durante o desenvolvimento
def calcular_ticket_medio(faturamento, pedidos):
    assert pedidos > 0, "não há pedidos para calcular ticket médio"
    assert faturamento >= 0, f"faturamento negativo: {faturamento}"
    return faturamento / pedidos


print(calcular_ticket_medio(10000, 25))

try:
    calcular_ticket_medio(10000, 0)
except AssertionError as e:
    print("AssertionError:", e)

> ⚠️ **`assert` não é validação de entrada.** Python remove todos os `assert` quando roda com a flag `-O` (otimizado). Use `assert` para checar pressupostos *seus*, durante o desenvolvimento; use `raise ValueError` para validar dados do usuário.

## 🔧 Prática guiada — Pipeline completo: CSV sujo → relatório

Este é o esqueleto do que você vai implementar no `projeto_Atlas`. Repare no fluxo: **ler → validar → transformar → agregar → gravar**, com erros registrados em vez de derrubar o programa.

In [ ]:
%%writefile pipeline_aurora.py
"""Pipeline mínimo: lê vendas de um CSV, valida, agrega e grava relatórios."""

import csv
import json
from collections import defaultdict
from pathlib import Path


class DadoInvalidoError(Exception):
    """Uma linha do CSV não pôde ser interpretada."""


def parse_linha(linha: dict, numero: int) -> dict:
    """Converte e valida uma linha do CSV. Levanta DadoInvalidoError."""
    try:
        registro = {
            "id": int(linha["id"]),
            "data": linha["data"].strip(),
            "cidade": linha["cidade"].strip().title(),
            "produto": linha["produto"].strip().title(),
            "quantidade": int(linha["quantidade"]),
            "preco_unitario": float(linha["preco_unitario"]),
            "status": linha["status"].strip().lower(),
        }
    except (ValueError, KeyError, AttributeError) as e:
        raise DadoInvalidoError(f"linha {numero}: {type(e).__name__}: {e}") from e

    if registro["quantidade"] <= 0:
        raise DadoInvalidoError(f"linha {numero}: quantidade inválida")
    if registro["preco_unitario"] < 0:
        raise DadoInvalidoError(f"linha {numero}: preço negativo")

    registro["total"] = round(registro["quantidade"] * registro["preco_unitario"], 2)
    return registro


def carregar(caminho: Path) -> tuple[list[dict], list[str]]:
    """Lê o CSV. Devolve (registros válidos, mensagens de erro)."""
    if not caminho.exists():
        raise FileNotFoundError(f"arquivo não encontrado: {caminho}")

    validos, erros = [], []
    with open(caminho, newline="", encoding="utf-8") as f:
        for numero, linha in enumerate(csv.DictReader(f), start=2):
            try:
                validos.append(parse_linha(linha, numero))
            except DadoInvalidoError as e:
                erros.append(str(e))
    return validos, erros


def agregar(registros: list[dict], campo: str) -> dict[str, float]:
    """Soma o total dos pedidos pagos agrupando por um campo."""
    acumulado = defaultdict(float)
    for r in registros:
        if r["status"] == "pago":
            acumulado[r[campo]] += r["total"]
    return {k: round(v, 2) for k, v in acumulado.items()}


def gravar_json(dados: dict, caminho: Path) -> None:
    caminho.parent.mkdir(parents=True, exist_ok=True)
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=2)


def main() -> None:
    entrada = Path("dados_aula/vendas.csv")
    saida = Path("dados_aula/saida/relatorio.json")

    try:
        registros, erros = carregar(entrada)
    except FileNotFoundError as e:
        print(f"❌ {e}")
        return

    por_cidade = agregar(registros, "cidade")
    por_produto = agregar(registros, "produto")

    relatorio = {
        "arquivo": str(entrada),
        "linhas_validas": len(registros),
        "linhas_rejeitadas": len(erros),
        "faturamento_total": round(sum(por_cidade.values()), 2),
        "por_cidade": por_cidade,
        "por_produto": por_produto,
        "erros": erros,
    }

    gravar_json(relatorio, saida)

    print(f"✅ {len(registros)} registros válidos, {len(erros)} rejeitados")
    print(f"💰 Faturamento: R$ {relatorio['faturamento_total']:,.2f}")
    print(f"📄 Relatório em: {saida}")
    for e in erros:
        print(f"   ⚠️  {e}")


if __name__ == "__main__":
    main()

In [ ]:
!python pipeline_aurora.py

In [ ]:
# Rodando contra o arquivo SUJO
import pipeline_aurora
from pathlib import Path

registros, erros = pipeline_aurora.carregar(Path("dados_aula/vendas_sujo.csv"))

print(f"✅ Válidos: {len(registros)}")
print(f"❌ Rejeitados: {len(erros)}\n")
for e in erros:
    print("  ", e)

print("\nFaturamento por cidade:", pipeline_aurora.agregar(registros, "cidade"))

## 📝 Exercícios rápidos

**E1.** Escreva `ler_int_seguro(texto, padrao=0)` que trate `ValueError` e `TypeError`. Teste com `"42"`, `"4.2"`, `""`, `None`, `"abc"`.

**E2.** Crie a exceção `CupomInvalidoError(AtlasError)` e uma função `aplicar_cupom(valor, codigo)` que aceite apenas `AURORA10` e `AURORA20`, levantando a exceção para qualquer outro código.

**E3.** Escreva `contar_linhas(caminho)` que devolva o número de linhas de um arquivo, tratando `FileNotFoundError` com uma mensagem amigável e devolvendo `0`.

**E4.** Leia `dados_aula/vendas.csv` e grave `dados_aula/pagos.csv` contendo apenas os pedidos pagos, com uma coluna extra `total`.

**E5.** Converta `dados_aula/vendas.csv` para `dados_aula/vendas.json` (lista de objetos), com tipos já convertidos.

**E6.** Escreva `carregar_config(caminho)` que leia um JSON e trate três casos: arquivo inexistente, JSON malformado e chave obrigatória ausente — cada um com mensagem específica.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

## ✅ Checklist de saída

- [ ] Sei ler um traceback de baixo para cima
- [ ] Conheço as exceções comuns e quando cada uma aparece
- [ ] Uso `try/except` com tipo **específico** e nunca `except: pass`
- [ ] Sei para que servem `else` e `finally`
- [ ] Levanto exceções com `raise` e crio exceções próprias
- [ ] Uso `pathlib.Path` em vez de concatenar strings
- [ ] **Sempre** uso `with open(...)` e passo `encoding="utf-8"`
- [ ] Leio CSV com `DictReader` e converto tipos explicitamente
- [ ] Trato linhas inválidas registrando o erro, sem derrubar o processo
- [ ] Leio e gravo JSON com `ensure_ascii=False`
- [ ] Sei colocar breakpoint, inspecionar variáveis e usar Step Into/Over no VS Code

---

### ➡️ Próxima etapa

**`01_99_Lista_Exercicios.ipynb`** — lista abrangente cobrindo todo o Módulo 01 + o enunciado do mini projeto **Relatório de Vendas Aurora**.